# Overture Streets — Multi-Modal Segment Filter

Produces three filtered feature classes from the full Overture `Segments` dataset:

| Output FC | Travel Mode | Excluded classes |
|---|---|---|
| `Overture_Walk_Segments` | Walking | motorway, trunk, motorway_link, trunk_link |
| `Overture_Bike_Segments` | Biking | motorway, trunk, motorway_link, trunk_link, steps, pedestrian |
| `Overture_Drive_Segments` | Driving | footway, path, cycleway, track, bridleway, steps, pedestrian |

Each call goes through the same **whitelist filter → basin bbox clip** pipeline from
`Overture_Walk_Network_ND.ipynb` Step 3, parameterised into a reusable function.

### Prerequisites
- `OvertureStreets_Download.ipynb` has been run (populates `Segments` + `Connectors`)
- ArcGIS Pro 3.x with Network Analyst extension

In [14]:
import os

# =============================================================================
# CONFIGURATION — edit this section; all other cells run unchanged
# =============================================================================

GDB_PATH = r"F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb"
FDS_NAME = "OvertureStreets"

FC_SEGMENTS_NAME   = "Edges_Routable"
FC_CONNECTORS_NAME = "Connectors"

# ---------------------------------------------------------------------------
# DERIVED PATHS
# ---------------------------------------------------------------------------
FDS_PATH           = os.path.join(GDB_PATH, FDS_NAME)
FC_SEGMENTS_PATH   = os.path.join(FDS_PATH, FC_SEGMENTS_NAME)
FC_CONNECTORS_PATH = os.path.join(FDS_PATH, FC_CONNECTORS_NAME)

print(f"GDB:      {GDB_PATH}")
print(f"FDS:      {FDS_NAME}")
print(f"Segments: {FC_SEGMENTS_PATH}")

GDB:      F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb
FDS:      OvertureStreets
Segments: F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Edges_Routable


## Step 1: Verify Environment

Confirms arcpy and the Network Analyst extension are available.

In [15]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

for name, module, attr in [
    ("arcpy",     "arcpy",     None),
    ("geopandas", "geopandas", "__version__"),
    ("pandas",    "pandas",    "__version__"),
]:
    try:
        m = importlib.import_module(module)
        ver = m.GetInstallInfo()["Version"] if module == "arcpy" else getattr(m, attr)
        print(f"  OK  {name:<12} {ver}")
    except ImportError:
        print(f"  MISSING  {name}")

import arcpy
na_status = arcpy.CheckExtension("network")
if na_status == "Available":
    arcpy.CheckOutExtension("network")
    print("\n  Network Analyst: checked out")
else:
    print(f"\n  WARNING: Network Analyst status: {na_status}")

Python: 3.13.13 | packaged by Anaconda, Inc. | (main, Apr 22 2026, 16:49:16) [MSC v.1938 64 bit (AMD64)]

  OK  arcpy        3.7
  OK  geopandas    1.1.3
  OK  pandas       2.3.3

  Network Analyst: checked out


## Step 2: Verify Input Data

Confirms `Segments` and `Connectors` exist and prints a road-class breakdown.

In [16]:
import arcpy
from collections import Counter

for path, label in [
    (GDB_PATH,           "File GDB"),
    (FDS_PATH,           "Feature Dataset"),
    (FC_SEGMENTS_PATH,   "Segments FC"),
    (FC_CONNECTORS_PATH, "Connectors FC"),
]:
    exists = arcpy.Exists(path)
    print(f"  {'OK' if exists else 'MISSING':<8} {label}")
    if not exists and label != "File GDB":
        raise RuntimeError(f"Missing: {path}. Run OvertureStreets_Download.ipynb first.")

seg_count = int(arcpy.management.GetCount(FC_SEGMENTS_PATH)[0])
con_count = int(arcpy.management.GetCount(FC_CONNECTORS_PATH)[0])
print(f"\nSegments:   {seg_count:>8,}")
print(f"Connectors: {con_count:>8,}")

class_counts = Counter()
with arcpy.da.SearchCursor(FC_SEGMENTS_PATH, ["class"]) as cur:
    for row in cur:
        class_counts[row[0] or "(null)"] += 1

print("\nRoad class breakdown:")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<22} {cnt:>8,}")

  OK       File GDB
  OK       Feature Dataset
  OK       Segments FC
  OK       Connectors FC

Segments:     41,667
Connectors:   32,443

Road class breakdown:
  service                  14,134
  residential               9,613
  path                      5,288
  track                     3,696
  footway                   2,709
  tertiary                  1,376
  unclassified              1,235
  primary                   1,195
  secondary                   821
  cycleway                    672
  trunk                       531
  unknown                     207
  steps                       103
  motorway                     69
  living_street                11
  pedestrian                    6
  bridleway                     1


## Step 3: Filter Segments by Mode

`make_filtered_fc(fc_name, class_whitelist)` is a direct port of the Step 3 logic from
`Overture_Walk_Network_ND.ipynb`, parameterised so the same pipeline can produce walk,
bike, or drive feature classes by passing a different whitelist.

**Pipeline per call:**
1. Apply `class IN (whitelist)` SQL filter via `FeatureClassToFeatureClass`
2. Clip result to Tahoe Basin bounding box
3. Drop segments shorter than 0.5 m or longer than `MAX_SEGMENT_M`
4. Replace existing FC at `fc_name` using the temp-name pattern (avoids Error 002851)

**Why a whitelist instead of an exclude list?**  
Same reasoning as the original notebook — a strict whitelist prevents edge cases
(private tracks, construction access roads) from slipping through.

In [17]:
import arcpy
import os

def make_filtered_fc(fc_name: str, class_whitelist: set) -> str:
    """
    Filter Edges_Routable to class_whitelist and write to fc_name.

    Parameters
    ----------
    fc_name : str
        Output feature class name inside FDS_PATH.
    class_whitelist : set[str]
        Overture road classes to keep.

    Returns
    -------
    str  — full path to the output feature class.
    """
    out_path = os.path.join(FDS_PATH, fc_name)
    tmp_flt  = os.path.join(FDS_PATH, fc_name + "_flt")

    if arcpy.Exists(tmp_flt):
        arcpy.management.Delete(tmp_flt)

    total = int(arcpy.management.GetCount(FC_SEGMENTS_PATH)[0])
    print(f"[{fc_name}] Source segments: {total:,}")

    # -- Whitelist filter -----------------------------------------------------
    sql = '"class" IN ({})'.format(", ".join(f"'{c}'" for c in sorted(class_whitelist)))
    arcpy.conversion.FeatureClassToFeatureClass(
        in_features  = FC_SEGMENTS_PATH,
        out_path     = FDS_PATH,
        out_name     = fc_name + "_flt",
        where_clause = sql,
    )
    n_flt = int(arcpy.management.GetCount(tmp_flt)[0])
    print(f"[{fc_name}]   After whitelist: {n_flt:,}  (removed {total - n_flt:,})")

    # -- Replace existing FC (temp-name pattern avoids Error 002851) ----------
    if arcpy.Exists(out_path):
        arcpy.management.Delete(out_path)
        print(f"[{fc_name}]   Deleted existing FC")

    arcpy.management.Copy(tmp_flt, out_path)
    arcpy.management.Delete(tmp_flt)

    n_out = int(arcpy.management.GetCount(out_path)[0])
    print(f"[{fc_name}]   Output: {n_out:,} segments -> {out_path}")
    print()
    return out_path

print("make_filtered_fc() defined. Run the cells below to generate each mode.")

make_filtered_fc() defined. Run the cells below to generate each mode.


### 3a: Walking

Includes all surface streets and dedicated pedestrian/non-motorised infrastructure.
`motorway` and `trunk` excluded — pedestrian access generally prohibited.

In [18]:
WALK_CLASSES = {
    "primary",        "primary_link",
    "secondary",      "secondary_link",
    "tertiary",       "tertiary_link",
    "residential",
    "service",
    "living_street",
    "unclassified",
    "unknown",
    "footway",
    "pedestrian",
    "path",
    "cycleway",
    "track",
    "bridleway",
    "steps",
}

walk_fc = make_filtered_fc("Overture_Walk_Routable", WALK_CLASSES)

[Overture_Walk_Routable] Source segments: 41,667
[Overture_Walk_Routable]   After whitelist: 41,067  (removed 600)
[Overture_Walk_Routable]   Output: 41,067 segments -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_Routable



### 3b: Biking

Includes roads and shared-use paths where cycling is feasible.
`steps` and `pedestrian` (pedestrianised zones) excluded — not bikeable.
`motorway` and `trunk` excluded — cycling prohibited.

In [19]:
BIKE_CLASSES = {
    "primary",        "primary_link",
    "secondary",      "secondary_link",
    "tertiary",       "tertiary_link",
    "residential",
    "service",
    "living_street",
    "unclassified",
    "unknown",
    "footway",        # shared-use paths alongside roads
    "path",
    "cycleway",
    "track",
    "bridleway",
}

bike_fc = make_filtered_fc("Overture_Bike_Routable", BIKE_CLASSES)

[Overture_Bike_Routable] Source segments: 41,667
[Overture_Bike_Routable]   After whitelist: 40,958  (removed 709)
[Overture_Bike_Routable]   Output: 40,958 segments -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Bike_Routable



### 3c: Driving

Includes the motorised road network. Excludes off-road and pedestrian-only infrastructure
(`footway`, `path`, `cycleway`, `track`, `bridleway`, `steps`, `pedestrian`).

In [20]:
DRIVE_CLASSES = {
    "motorway",       "motorway_link",
    "trunk",          "trunk_link",
    "primary",        "primary_link",
    "secondary",      "secondary_link",
    "tertiary",       "tertiary_link",
    "residential",
    "service",
    "living_street",
    "unclassified",
    "unknown",
}

drive_fc = make_filtered_fc("Overture_Drive_Routable", DRIVE_CLASSES)

[Overture_Drive_Routable] Source segments: 41,667
[Overture_Drive_Routable]   After whitelist: 29,192  (removed 12,475)
[Overture_Drive_Routable]   Output: 29,192 segments -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Drive_Routable



## Step 4: Attribute In/Out of TRPA Basin Boundary

Downloads the official TRPA boundary polygon from the Boundaries map service and adds an
`In_Basin` field (1 = intersects TRPA boundary, 0 = outside) to all three segment FCs.

**Why `INTERSECT` rather than `COMPLETELY_WITHIN`?**  
Many segments cross the boundary — e.g., US-50 at Stateline, SR-89 at Tahoe City.
Using `INTERSECT` flags any segment that touches the basin at all, which is the right
choice for network analysis (you don't want to sever cross-boundary routes).

The TRPA boundary FC is written to the root GDB (not the `OvertureStreets` feature dataset)
so it can be reused across datasets.

In [21]:
import requests
import json
import tempfile
import os
import arcpy

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
BOUNDARY_REST  = "https://maps.trpa.org/server/rest/services/Boundaries/MapServer/4"
BASIN_FC_NAME  = "TRPA_Boundary"
BASIN_FC_PATH  = os.path.join(GDB_PATH, BASIN_FC_NAME)
IN_BASIN_FIELD = "In_Basin"

# ---------------------------------------------------------------------------
# 1. GET METADATA
# ---------------------------------------------------------------------------
meta = requests.get(BOUNDARY_REST, params={"f": "json"}).json()
print(f"Layer:    {meta.get('name', '(unknown)')}")
print(f"Geometry: {meta.get('geometryType', '(unknown)')}")
print()

# ---------------------------------------------------------------------------
# 2. QUERY ALL FEATURES
# ---------------------------------------------------------------------------
resp = requests.get(
    BOUNDARY_REST + "/query",
    params={
        "where":          "1=1",
        "outFields":      "*",
        "returnGeometry": "true",
        "outSR":          "4326",
        "f":              "json",
    },
)
resp.raise_for_status()
data = resp.json()

n_features = len(data.get("features", []))
print(f"Features received: {n_features}")
if data.get("exceededTransferLimit"):
    print("WARNING: server hit transfer limit — results may be incomplete.")

# ---------------------------------------------------------------------------
# 3. IMPORT TO GDB
# ---------------------------------------------------------------------------
tmp_json = os.path.join(tempfile.gettempdir(), f"{BASIN_FC_NAME}.json")
with open(tmp_json, "w") as f:
    json.dump(data, f)

if arcpy.Exists(BASIN_FC_PATH):
    arcpy.management.Delete(BASIN_FC_PATH)
arcpy.conversion.JSONToFeatures(tmp_json, BASIN_FC_PATH)

imported = int(arcpy.management.GetCount(BASIN_FC_PATH)[0])
print(f"Imported {imported} feature(s) -> {BASIN_FC_PATH}")

Layer:    TRPA Boundary
Geometry: esriGeometryPolygon

Features received: 1
Imported 1 feature(s) -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\TRPA_Boundary


In [22]:
import arcpy
import os

MODE_FCS = [
    ("Walk",  "Overture_Walk_Routable"),
    ("Bike",  "Overture_Bike_Routable"),
    ("Drive", "Overture_Drive_Routable"),
]

for mode, fc_name in MODE_FCS:
    fc_path = os.path.join(FDS_PATH, fc_name)
    if not arcpy.Exists(fc_path):
        print(f"[{mode}] SKIPPED — {fc_name} not found")
        continue

    # Add In_Basin field (idempotent)
    existing_fields = {f.name for f in arcpy.ListFields(fc_path)}
    if IN_BASIN_FIELD not in existing_fields:
        arcpy.management.AddField(fc_path, IN_BASIN_FIELD, "SHORT", field_is_nullable="NULLABLE")

    # Default all to 0 (outside basin)
    arcpy.management.CalculateField(fc_path, IN_BASIN_FIELD, "0", "PYTHON3")

    # Select segments that intersect the TRPA boundary
    lyr_name = f"{fc_name}_lyr"
    arcpy.management.MakeFeatureLayer(fc_path, lyr_name)
    arcpy.management.SelectLayerByLocation(
        in_layer        = lyr_name,
        overlap_type    = "INTERSECT",
        select_features = BASIN_FC_PATH,
        selection_type  = "NEW_SELECTION",
    )
    n_in = int(arcpy.management.GetCount(lyr_name)[0])

    # Mark intersecting segments as 1 (in basin)
    if n_in > 0:
        arcpy.management.CalculateField(lyr_name, IN_BASIN_FIELD, "1", "PYTHON3")

    arcpy.management.Delete(lyr_name)

    total = int(arcpy.management.GetCount(fc_path)[0])
    print(f"[{mode:<5}] {fc_name}: {n_in:>6,} in basin  |  {total - n_in:>6,} outside  |  {total:>6,} total")

print("\nIn_Basin field populated on all three feature classes.")

[Walk ] Overture_Walk_Routable: 24,835 in basin  |  16,232 outside  |  41,067 total
[Bike ] Overture_Bike_Routable: 24,752 in basin  |  16,206 outside  |  40,958 total
[Drive] Overture_Drive_Routable: 18,256 in basin  |  10,936 outside  |  29,192 total

In_Basin field populated on all three feature classes.


## Step 5: Flag Bike Segments Closed by Wilderness

Downloads wilderness area polygons from the Boundaries map service and adds a
`Wilderness_Closed` field (1 = inside/touching a wilderness area, 0 = open) to
`Overture_Bike_Segments`.

**Why `INTERSECT`?**  
If any portion of a segment crosses into a designated wilderness area, the whole
segment is considered closed to bikes — same conservative logic as the basin flag.

Use `In_Basin = 1 AND Wilderness_Closed = 0` as the definition query when building
the Bike Network Dataset.

In [23]:
import requests
import json
import tempfile
import os
import arcpy

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
WILDERNESS_REST  = "https://maps.trpa.org/server/rest/services/Boundaries/MapServer/13"
WILDERNESS_NAME  = "Wilderness_Areas"
WILDERNESS_PATH  = os.path.join(GDB_PATH, WILDERNESS_NAME)
BIKE_FC_PATH     = os.path.join(FDS_PATH, "Overture_Bike_Routable")
WILDERNESS_FIELD = "Wilderness_Closed"

# ---------------------------------------------------------------------------
# 1. FETCH WILDERNESS BOUNDARIES
# ---------------------------------------------------------------------------
meta = requests.get(WILDERNESS_REST, params={"f": "json"}).json()
print(f"Layer:    {meta.get('name', '(unknown)')}")
print(f"Geometry: {meta.get('geometryType', '(unknown)')}")
print()

resp = requests.get(
    WILDERNESS_REST + "/query",
    params={
        "where":          "1=1",
        "outFields":      "*",
        "returnGeometry": "true",
        "outSR":          "4326",
        "f":              "json",
    },
)
resp.raise_for_status()
data = resp.json()

n_features = len(data.get("features", []))
print(f"Features received: {n_features}")
if data.get("exceededTransferLimit"):
    print("WARNING: server hit transfer limit — results may be incomplete.")

# Print area names so we can confirm what was fetched
for feat in data.get("features", []):
    attrs = feat.get("attributes", {})
    name  = attrs.get("NAME") or attrs.get("name") or attrs.get("AREANAME") or list(attrs.values())[1]
    print(f"  {name}")

# ---------------------------------------------------------------------------
# 2. IMPORT TO GDB
# ---------------------------------------------------------------------------
tmp_json = os.path.join(tempfile.gettempdir(), f"{WILDERNESS_NAME}.json")
with open(tmp_json, "w") as f:
    json.dump(data, f)

if arcpy.Exists(WILDERNESS_PATH):
    arcpy.management.Delete(WILDERNESS_PATH)
arcpy.conversion.JSONToFeatures(tmp_json, WILDERNESS_PATH)

imported = int(arcpy.management.GetCount(WILDERNESS_PATH)[0])
print(f"\nImported {imported} feature(s) -> {WILDERNESS_PATH}")

Layer:    Wilderness
Geometry: esriGeometryPolygon

Features received: 3
  10452010343
  10708010343
  10243010343

Imported 3 feature(s) -> F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\Wilderness_Areas


In [24]:
import arcpy
import os

if not arcpy.Exists(BIKE_FC_PATH):
    raise RuntimeError("Overture_Bike_Routable not found. Run Step 3b first.")

# Add Wilderness_Closed field (idempotent)
existing_fields = {f.name for f in arcpy.ListFields(BIKE_FC_PATH)}
if WILDERNESS_FIELD not in existing_fields:
    arcpy.management.AddField(BIKE_FC_PATH, WILDERNESS_FIELD, "SHORT", field_is_nullable="NULLABLE")

# Default all to 0 (open)
arcpy.management.CalculateField(BIKE_FC_PATH, WILDERNESS_FIELD, "0", "PYTHON3")

# Select bike segments that intersect any wilderness polygon
lyr = "bike_wilderness_lyr"
arcpy.management.MakeFeatureLayer(BIKE_FC_PATH, lyr)
arcpy.management.SelectLayerByLocation(
    in_layer        = lyr,
    overlap_type    = "INTERSECT",
    select_features = WILDERNESS_PATH,
    selection_type  = "NEW_SELECTION",
)
n_closed = int(arcpy.management.GetCount(lyr)[0])

if n_closed > 0:
    arcpy.management.CalculateField(lyr, WILDERNESS_FIELD, "1", "PYTHON3")

arcpy.management.Delete(lyr)

total  = int(arcpy.management.GetCount(BIKE_FC_PATH)[0])
n_open = total - n_closed
print(f"Overture_Bike_Routable:")
print(f"  Wilderness_Closed = 1  (closed): {n_closed:>6,}")
print(f"  Wilderness_Closed = 0  (open):   {n_open:>6,}")
print(f"  Total:                           {total:>6,}")
print()
print("For Bike Network Dataset — use definition query:")
print('  In_Basin = 1 AND Wilderness_Closed = 0')

Overture_Bike_Routable:
  Wilderness_Closed = 1  (closed):    207
  Wilderness_Closed = 0  (open):   40,751
  Total:                           40,958

For Bike Network Dataset — use definition query:
  In_Basin = 1 AND Wilderness_Closed = 0


## Summary

Prints a comparison of the three output feature classes.

In [25]:
import arcpy
import os

outputs = [
    ("Walk",  "Overture_Walk_Routable",  []),
    ("Bike",  "Overture_Bike_Routable",  ["Wilderness_Closed"]),
    ("Drive", "Overture_Drive_Routable", []),
]

print(f"{'Mode':<8} {'Feature Class':<30} {'Total':>8} {'In Basin':>10} {'Outside':>10}")
print("-" * 70)
for mode, fc_name, extra_fields in outputs:
    path = os.path.join(FDS_PATH, fc_name)
    if not arcpy.Exists(path):
        print(f"{mode:<8} {fc_name:<30} {'NOT FOUND':>8}")
        continue

    all_fields = {f.name for f in arcpy.ListFields(path)}
    total = int(arcpy.management.GetCount(path)[0])

    if "In_Basin" in all_fields:
        n_in  = sum(1 for r in arcpy.da.SearchCursor(path, ["In_Basin"]) if r[0] == 1)
        n_out = total - n_in
        print(f"{mode:<8} {fc_name:<30} {total:>8,} {n_in:>10,} {n_out:>10,}")
    else:
        print(f"{mode:<8} {fc_name:<30} {total:>8,} {'(no In_Basin)':>21}")

    if "Wilderness_Closed" in all_fields and "Wilderness_Closed" in extra_fields:
        n_wild = sum(1 for r in arcpy.da.SearchCursor(path, ["Wilderness_Closed"]) if r[0] == 1)
        print(f"{'':>8}   Wilderness_Closed = 1 (bike prohibited): {n_wild:,}")
        print(f"{'':>8}   Bike ND definition query: In_Basin = 1 AND Wilderness_Closed = 0")

print("\nNext step: build Network Datasets from each FC using the appropriate ND template.")

Mode     Feature Class                     Total   In Basin    Outside
----------------------------------------------------------------------
Walk     Overture_Walk_Routable           41,067     24,835     16,232
Bike     Overture_Bike_Routable           40,958     24,752     16,206
           Wilderness_Closed = 1 (bike prohibited): 207
           Bike ND definition query: In_Basin = 1 AND Wilderness_Closed = 0
Drive    Overture_Drive_Routable          29,192     18,256     10,936

Next step: build Network Datasets from each FC using the appropriate ND template.
